# Kimi K3 via the direct Moonshot API

Runs annotation through Moonshot's first-party endpoint instead of
OpenRouter. Why: first-party serving has no 64K completion cap
(`max_completion_tokens` defaults to 131,072, settable to 1M), full
precision, and automatic prefix caching that discounts the repeated P0
system prompt. Logic lives in
`extension/scripts/model_specific/moonshot_kimi.py`; cache, validation,
and scoring are shared. Records cache under
`moonshot-direct/kimi-k3-{effort}`, separate from OpenRouter K3 records.

K3 fixes sampling server-side (temperature 1.0, top_p 0.95), so this
notebook deliberately omits temperature rather than sending the requested
temperature 0 value that the endpoint does not support. K3 therefore remains
stochastic across runs on this backend.
Effort levels: low, high, max (max is the platform default and the
configuration the OpenRouter sweeps effectively ran).

**Before running:** put `MOONSHOT_API_KEY=...` in the environment or in
`.env` at the repo root. K3 unlocks after a minimum $1 top-up; rate
limits scale with the account tier, so keep workers modest.

P0 records are saved under
`extension/artifacts/extraction_cache/train/moonshot-direct__kimi-k3-max/P0/{dialogue_id}.json`.


In [1]:
import os, sys
from pathlib import Path
_here = Path.cwd()
for _c in [_here, *_here.parents]:
    if (_c / "extension" / "artifacts").exists():
        os.chdir(_c); break
sys.path.insert(0, str(Path.cwd()))
from extension.scripts.model_specific import moonshot_kimi
try:
    moonshot_kimi._api_key(); _key = True
except RuntimeError:
    _key = False
print("cwd:", os.getcwd(), "| MOONSHOT key found:", _key)


cwd: /Users/tandon.utsav2/Desktop/Experiment_1 | MOONSHOT key found: True


In [2]:
from extension.scripts.load_annotation_data import load_dataset
from extension.scripts import prompt_loader, extraction, scoring

gold = load_dataset("extension/artifacts/annotation_dev_and_val_sets/validation_set.csv")
DIALOGUES = extraction.dialogues_from(gold, split="train")
print(f"{len(DIALOGUES)} dialogues, {len(gold)} units")


78 dialogues, 544 units


In [3]:
TEST_PROMPTS = ['P6']
N_TEST_DIALOGUES = 78
MAX_WORKERS = 10
EFFORT = 'max'        # K3 scale: 'low', 'high', 'max' (platform default)


In [4]:
# TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
TEST_DIALOGUES = DIALOGUES[0:39]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}"
SLUG = moonshot_kimi.cache_slug(EFFORT)
print(f"moonshot-direct kimi-k3 effort={EFFORT} on "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x {TEST_PROMPTS}\n")

import json as _json
import pandas as pd
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
pooled_family_f1_cols = [f'pooled_f1_{family}' for family in scoring.POOLED_FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    moonshot_kimi.generate_annotations(prompt, TEST_DIALOGUES,
                                       reasoning_effort=EFFORT, max_workers=MAX_WORKERS)
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(SLUG, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        u = (rec['attempts'][0].get('meta') or {}).get('usage', {}) if rec['attempts'] else {}
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"completion {u.get('completion_tokens','?')} tok  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, SLUG, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split='train')
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
          f"| accuracy {s['accuracy']:.3f} | alpha {s['alpha']:.3f}")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ))
    print(f"     pooled: macro-F1(P) {s['pooled_macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['pooled_micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['pooled_weighted_f1_P']:.3f} "
          f"| accuracy {s['pooled_accuracy']:.3f} "
          f"| alpha {s['pooled_alpha']:.3f}")
    print("     pooled family F1(P): " + " | ".join(
        f"{family}={s[f'pooled_f1_{family}']:.3f}"
        for family in scoring.POOLED_FAMILIES
    ) + "\n")

results = pd.DataFrame(test_rows)
summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
                'weighted_f1_P', 'accuracy', 'alpha',
                *family_f1_cols, 'latency_s']
pooled_summary_cols = ['prompt', 'valid_rate', 'pooled_macro_f1_P',
                       'pooled_micro_f1_P', 'pooled_weighted_f1_P',
                       'pooled_accuracy', 'pooled_alpha',
                       *pooled_family_f1_cols, 'latency_s']
summary = results[summary_cols].round(3)
pooled_summary = results[pooled_summary_cols].round(3)
print('Original five-family scores')
print(summary.to_string(index=False))
print('\nPooled conceptual/procedural scores')
print(pooled_summary.to_string(index=False))


moonshot-direct kimi-k3 effort=max on [1, 21, 35, 79, 143, 178, 255, 270, 275, 289, 300, 306, 323, 344, 351, 356, 380, 434, 448, 494, 532, 554, 589, 617, 635, 656, 695, 736, 758, 779, 818, 822, 842, 862, 947, 958, 966, 980, 992] x ['P6']

  P6 (39 dialogues, 10 workers):
  143: ok
  255: ok
  289: ok
  35: ok
  21: ok
  178: ok
  79: ok
  275: ok
  356: ok
  300: ok
  344: ok
  270: ok
  1: ok
  306: ok
  351: ok
  448: ok
  494: ok
  532: ok
  380: ok
  554: ok
  323: ok
  617: ok
  434: ok
  695: ok
  635: ok
  656: ok
  818: ok
  842: ok
  589: ok
  779: ok
  822: ok
  736: ok
  947: ok
  958: ok
  758: ok
  992: ok
  980: ok
  862: ok
  966: ok
  P6                     1: ok       completion 26400 tok  latency 1399.4s
  P6                     21: ok       completion 12486 tok  latency 364.2s
  P6                     35: ok       completion 11620 tok  latency 337.1s
  P6                     79: ok       completion 19107 tok  latency 546.4s
  P6                     143: ok       comp

### Notes

Cost prints are omitted: the direct API reports tokens (with cache-hit
accounting on billing), not dollars. Prefix caching is automatic, so the
P0 system prompt should show as cache hits from the second
dialogue onward. Invalid records re-fire on the next execution; the purge
script covers this cache tree too. For a deadlock cell, raising
`moonshot_kimi.MAX_COMPLETION_TOKENS` (up to 1,048,576) buys headroom.
